# Compare TF-IDF (Baseline) vs TF-IDF with Translation

**Goal:** Measure whether translating all text to French before TF-IDF improves product classification.

**Setup:**
- Same train/val split (`random_state=42`, `test_size=0.20`, stratified)
- Same LightGBM hyperparameters
- Same metric: **weighted F1-score**

**Prerequisites:**
```bash
python models/baseline_model.py       # generates tfidf_vectorizer.pkl + lgb_text_model.pkl
python models/tfidf_translated.py     # generates tfidf_translated_vectorizer.pkl + lgb_translated_model.pkl
```

In [ ]:
import pickle
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, f1_score, confusion_matrix

## 1. Load Data & Reproduce Validation Split

In [ ]:
# Original cleaned data
train = pd.read_pickle("../data/train_cleaned.pkl")

le = LabelEncoder()
train["label"] = le.fit_transform(train["prdtypecode"])

X_train, X_val, y_train, y_val = train_test_split(
    train, train["label"],
    test_size=0.20, random_state=42, stratify=train["label"],
)

# Translated data (same rows, same split)
train_trans = pd.read_pickle("../data/train_translated.pkl")
train_trans["label"] = le.transform(train_trans["prdtypecode"])
_, X_val_trans, _, _ = train_test_split(
    train_trans, train_trans["label"],
    test_size=0.20, random_state=42, stratify=train_trans["label"],
)

classes = [str(c) for c in le.classes_]
print(f"Val set: {len(X_val):,} rows | {len(classes)} classes")

## 2. Load Models & Generate Predictions

In [ ]:
# ── A) Baseline TF-IDF ────────────────────────────────────────────────────
with open("../models/tfidf_vectorizer.pkl", "rb") as f:
    tfidf_base = pickle.load(f)
with open("../models/lgb_text_model.pkl", "rb") as f:
    model_base = pickle.load(f)

X_va_base = tfidf_base.transform(X_val["text_clean"])
y_pred_base = model_base.predict(X_va_base)
f1_base = f1_score(y_val, y_pred_base, average="weighted")

print(f"Baseline TF-IDF — weighted F1: {f1_base:.4f}")

In [ ]:
# ── B) Translated TF-IDF ──────────────────────────────────────────────────
with open("../models/tfidf_translated_vectorizer.pkl", "rb") as f:
    tfidf_trans = pickle.load(f)
with open("../models/lgb_translated_model.pkl", "rb") as f:
    model_trans = pickle.load(f)

X_va_trans = tfidf_trans.transform(X_val_trans["text_fr_clean"])
y_pred_trans = model_trans.predict(X_va_trans)
f1_trans = f1_score(y_val, y_pred_trans, average="weighted")

print(f"Translated TF-IDF — weighted F1: {f1_trans:.4f}")

## 3. Overall Comparison

In [ ]:
delta = f1_trans - f1_base

summary = pd.DataFrame({
    "Model": ["Baseline TF-IDF (multilingual)", "Translated TF-IDF (all French)"],
    "Weighted F1": [round(f1_base, 4), round(f1_trans, 4)],
    "Delta": ["—", f"{delta:+.4f}"],
})
print(summary.to_string(index=False))
print(f"\nResult: translation {'improves' if delta > 0 else 'does not improve'} weighted F1 by {abs(delta):.4f}")

## 4. Per-Class F1 Comparison

In [ ]:
report_base = classification_report(y_val, y_pred_base, target_names=classes, output_dict=True)
report_trans = classification_report(y_val, y_pred_trans, target_names=classes, output_dict=True)

f1_df = pd.DataFrame({
    "class": classes,
    "f1_baseline": [report_base[c]["f1-score"] for c in classes],
    "f1_translated": [report_trans[c]["f1-score"] for c in classes],
})
f1_df["delta"] = f1_df["f1_translated"] - f1_df["f1_baseline"]
f1_df = f1_df.sort_values("delta", ascending=False)

# Grouped bar: side-by-side F1 per class
fig = go.Figure()
fig.add_trace(go.Bar(
    name="Baseline", x=f1_df["class"], y=f1_df["f1_baseline"],
    marker_color="steelblue",
))
fig.add_trace(go.Bar(
    name="Translated", x=f1_df["class"], y=f1_df["f1_translated"],
    marker_color="coral",
))
fig.update_layout(
    barmode="group",
    title="Per-Class F1: Baseline vs Translated TF-IDF",
    xaxis_title="Product Category", yaxis_title="F1-score",
    height=500, yaxis_range=[0, 1],
)
fig.show()

## 5. Per-Class F1 Delta (Translated − Baseline)

In [ ]:
fig_delta = go.Figure(go.Bar(
    x=f1_df["class"],
    y=f1_df["delta"],
    marker_color=f1_df["delta"].apply(lambda d: "green" if d > 0 else "red"),
    text=f1_df["delta"].apply(lambda d: f"{d:+.3f}"),
    textposition="outside",
))
fig_delta.update_layout(
    title="Per-Class F1 Change (Translated − Baseline)<br><sup>Green = improved, Red = regressed</sup>",
    xaxis_title="Product Category", yaxis_title="F1 Delta",
    height=500,
)
fig_delta.show()

n_improved = (f1_df["delta"] > 0).sum()
n_regressed = (f1_df["delta"] < 0).sum()
n_unchanged = (f1_df["delta"] == 0).sum()
print(f"Categories improved: {n_improved} | regressed: {n_regressed} | unchanged: {n_unchanged}")

## 6. Top Improvers & Regressors

In [ ]:
print("Top 5 IMPROVED categories:")
print(f1_df.head(5)[["class", "f1_baseline", "f1_translated", "delta"]].to_string(index=False))
print(f"\nTop 5 REGRESSED categories:")
print(f1_df.tail(5)[["class", "f1_baseline", "f1_translated", "delta"]].to_string(index=False))

## 7. Confusion Matrix Comparison

In [ ]:
from plotly.subplots import make_subplots

cm_base = confusion_matrix(y_val, y_pred_base, normalize="true")
cm_trans = confusion_matrix(y_val, y_pred_trans, normalize="true")

fig_cm = make_subplots(
    rows=1, cols=2,
    subplot_titles=[
        f"Baseline TF-IDF (F1={f1_base:.4f})",
        f"Translated TF-IDF (F1={f1_trans:.4f})",
    ],
    horizontal_spacing=0.12,
)

fig_cm.add_trace(
    go.Heatmap(z=cm_base, x=classes, y=classes, colorscale="Blues",
               showscale=False, zmin=0, zmax=1),
    row=1, col=1,
)
fig_cm.add_trace(
    go.Heatmap(z=cm_trans, x=classes, y=classes, colorscale="Blues",
               showscale=True, zmin=0, zmax=1,
               colorbar=dict(title="Rate", x=1.02)),
    row=1, col=2,
)
fig_cm.update_yaxes(title_text="True", row=1, col=1)
fig_cm.update_xaxes(title_text="Predicted", row=1, col=1)
fig_cm.update_xaxes(title_text="Predicted", row=1, col=2)
fig_cm.update_layout(height=650, width=1300, title_text="Confusion Matrices — Side by Side")
fig_cm.show()

## 8. Classification Reports

In [ ]:
print("=" * 70)
print("BASELINE TF-IDF")
print("=" * 70)
print(classification_report(y_val, y_pred_base, target_names=classes))

print("=" * 70)
print("TRANSLATED TF-IDF")
print("=" * 70)
print(classification_report(y_val, y_pred_trans, target_names=classes))